# Graphs II — Shortest Paths, Flow & Spanning Trees

With **weights** on edges, traversal gives way to optimization: the cheapest route between vertices, the maximum throughput of a network, the minimum-cost tree connecting everything. These algorithms differ in what they assume (non-negative weights? a single source? a heuristic?) and in their cost. Each records snapshots so the relaxation, augmentation, or edge selection can be watched updating the graph.

$$ \delta(s,v) = \min_{\text{paths } s \to v} \sum_{e \in \text{path}} w(e). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import heapq
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=750)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

# Weighted directed graph (used both ways where noted). edges: (u, v, w)
EDGES = [('A','B',4),('A','C',2),('C','B',1),('B','D',5),
         ('C','D',8),('C','E',10),('D','E',2),('D','F',6),('E','F',3)]
VERTS = ['A','B','C','D','E','F']
COORD = {'A':(0,1),'B':(1.3,1.8),'C':(1.3,0.2),'D':(2.7,1.4),'E':(2.7,-0.2),'F':(4,0.7)}

def adj_list(edges, undirected=False):
    g = {v: [] for v in VERTS}
    for u, v, w in edges:
        g[u].append((v, w))
        if undirected: g[v].append((u, w))
    return g

def draw_weighted(dist=None, current=None, settled=(), active_edges=(), tree_edges=(),
                  title='', extra='', directed=True, edge_labels=None):
    fig, ax = plt.subplots(figsize=(8.5, 5))
    elabels = edge_labels if edge_labels is not None else {(u,v): w for u,v,w in EDGES}
    for (u, v), w in elabels.items():
        x0,y0 = COORD[u]; x1,y1 = COORD[v]
        in_tree = (u,v) in tree_edges or (v,u) in tree_edges
        in_active = (u,v) in active_edges or (v,u) in active_edges
        col = 'seagreen' if in_tree else ('tomato' if in_active else 'lightgray')
        lw = 3 if (in_tree or in_active) else 1.3
        if directed:
            ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                        arrowprops=dict(arrowstyle='-|>', lw=lw, color=col,
                                        shrinkA=14, shrinkB=14))
        else:
            ax.plot([x0,x1],[y0,y1], color=col, lw=lw, zorder=1)
        mx,my = (x0+x1)/2, (y0+y1)/2
        ax.text(mx, my, str(w), fontsize=9, color='dimgray',
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'))
    for v in VERTS:
        x,y = COORD[v]
        if v == current:   c='tomato'
        elif v in settled: c='seagreen'
        else:              c='lightsteelblue'
        ax.add_patch(plt.Circle((x,y),0.2, facecolor=c, edgecolor='k', zorder=2))
        lbl = v if dist is None else f'{v}\n{"inf" if dist[v]==float("inf") else dist[v]}'
        ax.text(x, y, lbl, ha='center', va='center', fontsize=9, zorder=3)
    ax.text(0.02, 0.02, extra, transform=ax.transAxes, fontsize=10, color='dimgray')
    xs=[p[0] for p in COORD.values()]; ys=[p[1] for p in COORD.values()]
    ax.set_xlim(min(xs)-0.6,max(xs)+0.6); ax.set_ylim(min(ys)-0.7,max(ys)+0.7)
    ax.set_title(title); ax.axis('off'); plt.show()

## Dijkstra — Greedy with a Priority Queue

Dijkstra grows a set of **settled** vertices, each time extracting the unsettled vertex of smallest tentative distance and **relaxing** its outgoing edges. It is correct only when weights are **non-negative**, because settling assumes no cheaper path can later appear. Cost with a binary heap is $O((|V|+|E|)\log|V|)$. Watch distances drop as edges relax and vertices settle green.

$$ \text{relax}(u,v):\ \text{if } d[u]+w(u,v) < d[v]\ \text{then } d[v] \leftarrow d[u]+w(u,v). $$

In [2]:
def dijkstra_frames(edges, src):
    g = adj_list(edges); dist = {v: float('inf') for v in VERTS}; dist[src]=0
    settled=set(); pq=[(0, src)]; tree={}
    frames=[(dict(dist), None, set(settled), set(), f'init: d[{src}]=0, others inf')]
    while pq:
        d,u = heapq.heappop(pq)
        if u in settled: continue
        settled.add(u)
        frames.append((dict(dist), u, set(settled), set(), f'settle {u} (d={d})'))
        for v,w in g[u]:
            if dist[u]+w < dist[v]:
                dist[v]=dist[u]+w; tree[v]=u; heapq.heappush(pq,(dist[v],v))
                frames.append((dict(dist), u, set(settled), {(u,v)},
                               f'relax {u}->{v}: d[{v}] = {dist[v]}'))
    frames.append((dict(dist), None, set(settled), set(), 'all shortest distances final'))
    return frames

dsrc = widgets.Dropdown(options=VERTS, value='A', description='source')
dij_area = widgets.Output()
def relaunch_dij(*_):
    frames = dijkstra_frames(EDGES, dsrc.value)
    def draw(k):
        dist,cur,settled,active,note = frames[k]
        draw_weighted(dist=dist, current=cur, settled=settled-({cur} if cur else set()),
                      active_edges=active, title=f'Dijkstra step {k}/{len(frames)-1}: {note}')
    dij_area.clear_output(wait=True)
    with dij_area: make_player(len(frames), draw)
dsrc.observe(relaunch_dij, 'value')
display(dsrc, dij_area)
relaunch_dij()

Dropdown(description='source', options=('A', 'B', 'C', 'D', 'E', 'F'), value='A')

Output()

## Bellman-Ford — Relax Everything, $|V|-1$ Times

Bellman-Ford drops Dijkstra's non-negativity requirement: it relaxes **all** edges repeatedly for $|V|-1$ passes, since a shortest path has at most $|V|-1$ edges. A further pass that still relaxes signals a **negative cycle**. Cost is $O(|V|\cdot|E|)$ — slower, but it handles negative weights. Step through pass by pass and watch distances settle.

$$ \text{after } |V|-1 \text{ passes, } d[v]=\delta(s,v); \quad \text{a relax on pass } |V| \Rightarrow \text{negative cycle.} $$

In [3]:
# add an optional negative edge to demonstrate
def bellman_frames(edges, src):
    dist={v:float('inf') for v in VERTS}; dist[src]=0
    frames=[(dict(dist), None, set(), f'init d[{src}]=0')]
    n=len(VERTS)
    for p in range(1, n):
        changed=False
        for u,v,w in edges:
            if dist[u]+w < dist[v]:
                dist[v]=dist[u]+w; changed=True
                frames.append((dict(dist), (u,v), set(), f'pass {p}: relax {u}->{v}, d[{v}]={dist[v]}'))
        if not changed:
            frames.append((dict(dist), None, set(), f'pass {p}: no change, converged early'))
            break
    frames.append((dict(dist), None, set(VERTS), 'done'))
    return frames

neg_chk = widgets.Checkbox(value=False, description='include negative edge C->B = -3')
bsrc = widgets.Dropdown(options=VERTS, value='A', description='source')
bf_area = widgets.Output()
def relaunch_bf(*_):
    edges = [e for e in EDGES if not (e[0]=='C' and e[1]=='B')]
    edges.append(('C','B',-3 if neg_chk.value else 1))
    frames = bellman_frames(edges, bsrc.value)
    def draw(k):
        dist,active,settled,note = frames[k]
        draw_weighted(dist=dist, active_edges={active} if active else set(),
                      settled=settled, title=f'Bellman-Ford step {k}/{len(frames)-1}: {note}',
                      edge_labels={(u,v):w for u,v,w in edges})
    bf_area.clear_output(wait=True)
    with bf_area: make_player(len(frames), draw)
neg_chk.observe(relaunch_bf,'value'); bsrc.observe(relaunch_bf,'value')
display(widgets.HBox([bsrc, neg_chk]), bf_area)
relaunch_bf()

Output()

## Floyd-Warshall — All-Pairs via Intermediate Vertices

Floyd-Warshall computes shortest distances between **every** pair by considering each vertex $k$ as a possible intermediate and asking whether routing through $k$ shortens any pair. Three nested loops give $O(|V|^3)$, independent of edge count — ideal for dense graphs. The heatmap updates as each intermediate $k$ is admitted.

$$ d_{ij}^{(k)} = \min\!\left(d_{ij}^{(k-1)},\ d_{ik}^{(k-1)} + d_{kj}^{(k-1)}\right). $$

In [4]:
def floyd_frames(edges):
    n=len(VERTS); idx={v:i for i,v in enumerate(VERTS)}
    INF=float('inf')
    D=[[0 if i==j else INF for j in range(n)] for i in range(n)]
    for u,v,w in edges: D[idx[u]][idx[v]]=min(D[idx[u]][idx[v]], w)
    frames=[([row[:] for row in D], None, 'initial direct edges')]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k]+D[k][j] < D[i][j]:
                    D[i][j]=D[i][k]+D[k][j]
        frames.append(([row[:] for row in D], k, f'allow intermediate {VERTS[k]}'))
    return frames

frames_fw = floyd_frames(EDGES)
def draw_fw(k):
    D, kk, note = frames_fw[k]
    n=len(VERTS)
    M=np.array([[ (np.nan if x==float('inf') else x) for x in row] for row in D], dtype=float)
    fig, ax = plt.subplots(figsize=(6.5,5.5))
    im=ax.imshow(M, cmap='viridis_r')
    ax.set_xticks(range(n)); ax.set_xticklabels(VERTS)
    ax.set_yticks(range(n)); ax.set_yticklabels(VERTS)
    for i in range(n):
        for j in range(n):
            val = D[i][j]
            ax.text(j,i,'inf' if val==float('inf') else int(val), ha='center', va='center',
                    color='white', fontsize=9)
    if kk is not None:
        ax.add_patch(plt.Rectangle((kk-0.5,-0.5),1,n, fill=False, edgecolor='tomato', lw=2))
        ax.add_patch(plt.Rectangle((-0.5,kk-0.5),n,1, fill=False, edgecolor='tomato', lw=2))
    ax.set_title(f'Floyd-Warshall step {k}/{len(frames_fw)-1}: {note}')
    ax.set_xlabel('to'); ax.set_ylabel('from'); plt.tight_layout(); plt.show()
make_player(len(frames_fw), draw_fw)

Output()

## A* — Dijkstra Guided by a Heuristic

A* speeds up single-target search by ordering the frontier on $f(v)=g(v)+h(v)$, where $g$ is the cost so far and $h$ is an **admissible** heuristic (never overestimating the remaining cost). With $h=$ straight-line distance to the goal, the search is pulled toward the target, settling far fewer vertices than Dijkstra. Watch the $f$-values guide expansion toward the goal.

$$ f(v) = g(v) + h(v), \qquad h \text{ admissible} \Rightarrow A^* \text{ finds the optimum.} $$

In [5]:
def euclid(a, b):
    (x0,y0),(x1,y1)=COORD[a],COORD[b]; return np.hypot(x1-x0, y1-y0)

def astar_frames(edges, src, goal):
    g_und = adj_list(edges, undirected=True)
    g={v:float('inf') for v in VERTS}; g[src]=0
    settled=set(); pq=[(euclid(src,goal),0,src)]
    frames=[(dict(g), None, set(settled), set(), f'start at {src}, goal {goal}')]
    while pq:
        f,gc,u = heapq.heappop(pq)
        if u in settled: continue
        settled.add(u)
        frames.append((dict(g), u, set(settled), set(),
                       f'expand {u}: g={gc}, h={euclid(u,goal):.1f}, f={f:.1f}'))
        if u==goal:
            frames.append((dict(g), None, set(settled), set(), f'goal {goal} reached')); break
        for v,w in g_und[u]:
            if g[u]+w < g[v]:
                g[v]=g[u]+w
                heapq.heappush(pq,(g[v]+euclid(v,goal), g[v], v))
                frames.append((dict(g), u, set(settled), {(u,v)},
                               f'relax {u}->{v}: g[{v}]={g[v]}, f={g[v]+euclid(v,goal):.1f}'))
    return frames

asrc=widgets.Dropdown(options=VERTS, value='A', description='source')
agoal=widgets.Dropdown(options=VERTS, value='F', description='goal')
ast_area=widgets.Output()
def relaunch_ast(*_):
    frames=astar_frames(EDGES, asrc.value, agoal.value)
    def draw(k):
        gd,cur,settled,active,note=frames[k]
        draw_weighted(dist=gd, current=cur, settled=settled-({cur} if cur else set()),
                      active_edges=active, directed=False,
                      edge_labels={tuple(sorted((u,v))):w for u,v,w in EDGES},
                      title=f'A* step {k}/{len(frames)-1}: {note}',
                      extra=f'node labels show g(v); ordered by f=g+h')
    ast_area.clear_output(wait=True)
    with ast_area: make_player(len(frames), draw)
asrc.observe(relaunch_ast,'value'); agoal.observe(relaunch_ast,'value')
display(widgets.HBox([asrc, agoal]), ast_area)
relaunch_ast()

Output()

## Ford-Fulkerson — Augmenting Paths for Max Flow

Treating weights as **capacities**, Ford-Fulkerson repeatedly finds a path from source to sink with spare capacity (an **augmenting path**) and pushes the bottleneck amount along it, updating a **residual graph**. When no augmenting path remains, the total pushed equals the **max flow**, which equals the **min cut**. Watch each augmenting path saturate and the flow total climb.

$$ |f| = \sum_{v} f(s,v), \qquad \text{max-flow} = \text{min-cut (Ford-Fulkerson theorem).} $$

In [6]:
def ff_frames(edges, src, sink):
    cap={}
    for u,v,w in edges:
        cap[(u,v)] = cap.get((u,v),0)+w
        cap.setdefault((v,u),0)
    flow=0; frames=[]
    def bfs_path():
        from collections import deque
        parent={src:None}; q=deque([src])
        while q:
            u=q.popleft()
            for v in VERTS:
                if v not in parent and cap.get((u,v),0)>0:
                    parent[v]=u
                    if v==sink:
                        path=[]; x=sink
                        while x is not None: path.append(x); x=parent[x]
                        return path[::-1]
                    q.append(v)
        return None
    frames.append((dict(cap), [], flow, 'residual = original capacities'))
    while True:
        path=bfs_path()
        if path is None:
            frames.append((dict(cap), [], flow, f'no augmenting path: max flow = {flow}')); break
        bottleneck=min(cap[(path[i],path[i+1])] for i in range(len(path)-1))
        edges_on=[(path[i],path[i+1]) for i in range(len(path)-1)]
        frames.append((dict(cap), edges_on, flow,
                       f'augment along {"->".join(path)}, bottleneck {bottleneck}'))
        for i in range(len(path)-1):
            a,b=path[i],path[i+1]; cap[(a,b)]-=bottleneck; cap[(b,a)]=cap.get((b,a),0)+bottleneck
        flow+=bottleneck
        frames.append((dict(cap), edges_on, flow, f'pushed {bottleneck}; total flow {flow}'))
    return frames

fsrc=widgets.Dropdown(options=VERTS, value='A', description='source')
fsink=widgets.Dropdown(options=VERTS, value='F', description='sink')
ff_area=widgets.Output()
def relaunch_ff(*_):
    frames=ff_frames(EDGES, fsrc.value, fsink.value)
    def draw(k):
        cap,path_edges,flow,note=frames[k]
        labels={(u,v):cap[(u,v)] for u,v,_ in EDGES}  # show residual capacity on forward edges
        draw_weighted(active_edges=set(path_edges), title=f'Ford-Fulkerson step {k}/{len(frames)-1}',
                      extra=f'{note}', edge_labels=labels, directed=True)
    ff_area.clear_output(wait=True)
    with ff_area: make_player(len(frames), draw)
fsrc.observe(relaunch_ff,'value'); fsink.observe(relaunch_ff,'value')
display(widgets.HBox([fsrc, fsink]), ff_area)
relaunch_ff()

Output()

## Kruskal — Minimum Spanning Tree by Sorted Edges

Kruskal builds a **minimum spanning tree** by sorting all edges ascending and adding each edge that does not form a **cycle**, detected with a **union-find** structure. It stops once $|V|-1$ edges are chosen. Cost is dominated by the sort, $O(|E|\log|E|)$. Watch edges be accepted (green) or rejected (would-cycle) in weight order.

$$ \text{MST weight} = \min_{\text{spanning trees } T} \sum_{e \in T} w(e), \qquad |T| = |V|-1. $$

In [7]:
def kruskal_frames(edges):
    parent={v:v for v in VERTS}
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        ra,rb=find(a),find(b)
        if ra==rb: return False
        parent[ra]=rb; return True
    order=sorted(edges, key=lambda e:e[2])
    tree=[]; total=0; frames=[]
    frames.append((list(tree), None, None, 0, f'edges sorted by weight: ' +
                   ', '.join(f'{u}{v}:{w}' for u,v,w in order)))
    for u,v,w in order:
        if find(u)!=find(v):
            union(u,v); tree.append((u,v)); total+=w
            frames.append((list(tree), (u,v), 'accept', total, f'add {u}-{v} ({w}); total {total}'))
        else:
            frames.append((list(tree), (u,v), 'reject', total, f'skip {u}-{v} ({w}): would form cycle'))
        if len(tree)==len(VERTS)-1:
            frames.append((list(tree), None, None, total, f'MST complete, weight {total}')); break
    return frames

frames_kr = kruskal_frames(EDGES)
def draw_kr(k):
    tree, cand, status, total, note = frames_kr[k]
    active = {cand} if (cand and status=='reject') else set()
    draw_weighted(tree_edges=set(tree), active_edges=active, directed=False,
                  edge_labels={tuple(sorted((u,v))):w for u,v,w in EDGES},
                  title=f'Kruskal step {k}/{len(frames_kr)-1}', extra=note)
make_player(len(frames_kr), draw_kr)

Output()